In [2]:
''' getting important libraries which is used during our Model building
    from sklearn.ensemble  we imported RandomForestRegressor,
    and then we evaluted our model using various metrics from sklearn.metrics like
    r2_score,mean_absolute_error,mean_squared_Error'''
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [5]:
'''making a DF pipeline to our cleaned data set
   known as 'delhivery_cleaned_with_chokes.csv' 
'''
df=pd.read_csv('delhivery_cleaned_1.csv')

In [3]:
''' in selected columns we had only those columns which has high correlation
    with our target columns 'log_segment_actual_time' 
    since our data is already divided into 'training' and 'test'  we creating
    train_df and test_df base on the logic that column df['data'] == 'training'
    data frame column == training for train_df and test_df=='test'
    similiarly  we getting out X_test and y_test , X_train and Y_train 
    X_train,X_test is selected using selected_features 
'''
selected_features = ['cutoff_factor','actual_distance_to_destination','osrm_time','osrm_distance','segment_osrm_distance','actual_distance_to_destination',]
target_col = 'segment_actual_time'
train_df = df[df['data'] == 'training'].copy()
test_df = df[df['data'] =='test'].copy()
X_train = train_df[selected_features]
y_train = train_df['segment_actual_time']
X_test = test_df[selected_features]
y_test = test_df['segment_actual_time']
''' Since train test split is done then
    now we building our model .
    we have used Random Forest.
    because it is the most baseline and basic model which can be possibly made without
    doing any extra hustle for 
    Logistics data is notoriously messy. we have to mix continuous numbers ( distance) 
    with categorical labels (Vehicle Type, Hub ID), and deal with extreme outliers 
    like a truck breaking down or a massive unexpected highway closure.
    The Reality: One outlier can heavily warp a linear regression line, throwing off normal predictions.
    The Advantage: Random Forest handles a mix of numerical and categorical data perfectly without requiring tedious scaling.
    More importantly, because it averages the predictions of hundreds of independent trees (bagging), 
    it is highly resilient to outliers and noise. A single anomalous breakdown won't derail the entire model's accuracy for standard shipments.
'''

rf_model = RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1)
rf_model.fit(X_train,y_train)
y_pred = rf_model.predict(X_test)
''' we thus then predicted our ETA using first by fitting our 
    training data then by predicting
    rf_model.predict(X_test) 
    we then then predicted
'''
#Metrics
''' we then check our goodness of fit using various metrices like
    mean_absolute_error,mean_squared_error,root_mean_squared_error,
    r2_score and printing them.
'''
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
r2 = r2_score(y_test,y_pred)
print("--- Random Forest Segment-Level Performance (Minutes) ---")
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")
#Trip level Accuracy.
''' we thus calculated our trip level performnace
    we first created a evaluting DataFrame 
    and then a trip df which contain groupby data based trip id basically to add
    the segment wise data and reset_index for row to be in 0,1,2,3,4....
'''
eval_df = pd.DataFrame({
    'trip_id': test_df['trip_uuid'],
    'actual_time':y_test,
    'predicted':y_pred,
    }
)
trip_df = eval_df.groupby('trip_id')[['actual_time','predicted']].sum().reset_index()
trip_df['lower_bound'] = trip_df['actual_time']*0.85
trip_df['upper_bound'] = trip_df['actual_time']*1.15
trip_df['within_15_percent'] = trip_df['predicted'].between(trip_df['lower_bound'],trip_df['upper_bound'])
''' we calculating our percentage of trips that has achieved between
    15_percent and then we finally printing Percentage of trips where 
    ETA is within actual_time ±15%
'''
percentage_accurate_trips = trip_df['within_15_percent'].mean() * 100
print("\n--- Random Forest Trip-Level Performance ---")
print(f"Total Unique Trips Evaluated: {len(trip_df)}")
print(f"Percentage of trips where ETA is within actual_time ±15%: {percentage_accurate_trips:.2f}%")
importance = rf_model.feature_importances_
''' we thus printing the features wise 
    importance of each feature'''
for col,imp in zip(selected_features,importance):
    print(f'{col}:{imp:.5f}')


--- Random Forest Segment-Level Performance (Minutes) ---
R² Score: 0.3751
Mean Absolute Error (MAE): 14.54 minutes
Root Mean Squared Error (RMSE): 44.41 minutes

--- Random Forest Trip-Level Performance ---
Total Unique Trips Evaluated: 4163
Percentage of trips where ETA is within actual_time ±15%: 35.82%
cutoff_factor:0.07908
actual_distance_to_destination:0.10493
osrm_time:0.14748
osrm_distance:0.16701
segment_osrm_distance:0.40088
actual_distance_to_destination:0.10062


In [ ]:
''' Trip level Peformance is that
    Total_unique trips that are being evaluted is 4163 
    and Percentage of trips where ETA(Expected Time of arrival) is within actual_time ±15%: 35.8%
    which means that out of 100 trips approximately we predicte 36 of them correctly which is slighly less than
    CLRM
'''